In [1]:
import torch, torchvision
import mmdet
import mmcv
import mmocr
import json

In [2]:
from mmcv import Config
cfg = Config.fromfile('/home/erik/Riksarkivet/Projects/fastighet/models/configs/satrn_plus.py')

In [3]:
from mmocr.apis import init_detector, model_inference
from mmocr.datasets.pipelines import MultiRotateAugOCR

#img_p = '/home/erik/Riksarkivet/Projects/fastighet/data/htr_training_set_40_batches/10001439/10001439_00000060.tif'
checkpoint = '/home/erik/Riksarkivet/Projects/fastighet/models/satrn_htr_million_plus_test_height_resume1/epoch_3.pth'
#out_file = 'outputs/1036169.jpg'

model = init_detector(cfg, checkpoint, device="cuda:0")
if model.cfg.data.test['type'] == 'ConcatDataset':
    model.cfg.data.test.pipeline = model.cfg.data.test['datasets'][0].pipeline

Use load_from_local loader


In [4]:
with open('/home/erik/Riksarkivet/Projects/fastighet/output/jsonex.json', 'r') as f:
    result_json = json.load(f)

In [24]:
print(result_json['10001009']['10001009_00000005'][0]['pred_conf'])

0.9395896862534916


In [5]:
%%time

import os
from glob import glob
from pathlib import Path

dataset_path = '/home/erik/Riksarkivet/Projects/fastighet/data/htr_test_set_1_million'

batches = glob(os.path.join(dataset_path, '*/'))
batches.sort()

for i, batch in enumerate(batches):

    batch_name = Path(batch).name
    
    imgs = glob(os.path.join(batch, '**'))
    imgs.sort()

    for img in imgs:
        property_index = int(Path(img).stem.split('_')[2])
        img_name = '_'.join(Path(img).stem.split('_')[0:2])
        result = model_inference(model, img)
        try:
            result_json[batch_name][img_name][property_index]['pred'] = result['text']
            result_json[batch_name][img_name][property_index]['pred_conf'] = str(result['score'])
        except:
            print(img)
    
    print(str(i) + ': ' + batch_name)
        

0: 10001009
1: 10001059
2: 10001239
3: 10001419
4: 10001429
5: 10001490
6: 10001510
7: 10001722
8: 10001933
9: 10002043
10: 10002390
11: 10002455
12: 10003120
13: 10003246
14: 10003296
15: 10003466
16: 10003654
17: 10003664
18: 10003724
19: 10003774
20: 10003814
21: 10003833
22: 10003940
23: 10004019
24: 10004241
25: 10004561
26: 10004621
27: 10004652
28: 10004821
29: 10004952
30: 10004992
31: 10005266
32: 10005359
33: 10005379
34: 10005399
35: 10005454
36: 10005582
37: 10005617
38: 10005632
39: 10005662
40: 10005759
41: 10005790
42: 10005922
43: 10006062
44: 10006092
45: 10006122
46: 10006215
47: 10006318
48: 10006684
49: 10006704
50: 10006815
51: 10006855
52: 10006957
53: 10007012
54: 10007019
55: 10007051
56: 10007114
57: 10007145
58: 10007160
59: 10007284
60: 10007427
61: 10007783
62: 10007843
63: 10007947
64: 10008023
65: 10008152
66: 10008171
67: 10008183
68: 10008442
69: 10008447
70: 10008457
71: 10008562
72: 10008592
73: 10008667
74: 10008697
75: 10008732
76: 10009149
77: 10009

In [6]:
import json

with open('/home/erik/Riksarkivet/Projects/fastighet/output/iteration3_htr_epoch3.json', 'w', encoding='utf8') as f:
    j = json.dumps(result_json, indent = 4, ensure_ascii=False)
    f.write(str(j))